# ChemicalEntity ↔ Tissue Relation-Wise Merge

Merges Chemical–Tissue triples from EvoAGE; resolves tissue names via BTO;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import pandas as pd

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'


# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/CHEMICALENTITY_TISSUE/ALL_CHEMICALENTITY_TISSUE.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionary — BTO (Tissue)

In [2]:
BTO = pd.read_csv(DB_DIR + 'bto/BTO_ALL_COMBINED.csv')
BTO_dict = dict(zip(BTO['ID'], BTO['name']))
print(f"BTO dict: {len(BTO_dict):,} entries")

BTO dict: 6,566 entries


## 2. Load KG Sources

### 2a. Metaboage

In [7]:
Metaboage = pd.read_csv(PROC_DIR + 'MetaboAge/MetaboAge_Human_Chem_Tissue.csv')
Metaboage.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
Metaboage.columns =Metaboage.columns.str.lower()
print(f"EvoAGE: {len(Metaboage):,} rows | columns: {list(Metaboage.columns)}")
Metaboage['head'] = Metaboage['head'].astype('Int64')

Metaboage['kg_type'] = 'Aging'
Metaboage['kg_source'] = 'MetaboAge'
Metaboage['species'] = 'Homo sapiens'
Metaboage

EvoAGE: 264 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'relation_source', 'species', 'head_alt_name', 'head_detail_name', 'head_smile_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,relation_source,species,head_alt_name,head_detail_name,head_smile_name,tail_detail_name,head_id_is,tail_id_is,kg_type,kg_source
0,3845,ChemicalEntity_Tissue,BTO:0000237,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,kynurenic acid,4-oxo-1H-quinoline-2-carboxylic acid,C1=CC=C2C(=C1)C(=O)C=C(N2)C(=O)O,cerebrospinal fluid,Pubchem,BTO_ID,Aging,MetaboAge
1,5280450,ChemicalEntity_Tissue,BTO:0001253,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,linoleic acid,"(9Z,12Z)-octadeca-9,12-dienoic acid",CCCCC/C=C\C/C=C\CCCCCCCC(=O)O,skin,Pubchem,BTO_ID,Aging,MetaboAge
2,124886,ChemicalEntity_Tissue,BTO:0000089,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,glutathione,(2S)-2-amino-5-[[(2R)-1-(carboxymethylamino)-1...,C(CC(=O)N[C@@H](CS)C(=O)NCC(=O)O)[C@@H](C(=O)O)N,blood,Pubchem,BTO_ID,Aging,MetaboAge
3,64960,ChemicalEntity_Tissue,BTO:0000089,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,"1,5-anhydroglucitol","(2R,3S,4R,5S)-2-(hydroxymethyl)oxane-3,4,5-triol",C1[C@@H]([C@H]([C@@H]([C@H](O1)CO)O)O)O,blood,Pubchem,BTO_ID,Aging,MetaboAge
4,<NA>,ChemicalEntity_Tissue,BTO:0000089,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,dimethyl-guanosine,NaN,NaN,blood,Pubchem,BTO_ID,Aging,MetaboAge
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,892,ChemicalEntity_Tissue,BTO:0001419,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,scyllo-inositol,"cyclohexane-1,2,3,4,5,6-hexol",C1(C(C(C(C(C1O)O)O)O)O)O,urine,Pubchem,BTO_ID,Aging,MetaboAge
260,6213,ChemicalEntity_Tissue,BTO:0001419,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,dimethyl sulfone,methylsulfonylmethane,CS(=O)(=O)C,urine,Pubchem,BTO_ID,Aging,MetaboAge
261,5571,ChemicalEntity_Tissue,BTO:0001419,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,N-methylnicotinic acid,1-methylpyridin-1-ium-3-carboxylic acid,C[N+]1=CC=CC(=C1)C(=O)O,urine,Pubchem,BTO_ID,Aging,MetaboAge
262,440810,ChemicalEntity_Tissue,BTO:0001419,ChemicalEntity,Tissue,MetaboAge,Homo sapiens,N-methyl-4-pyridone-3-carboxamide,1-methyl-4-oxopyridine-3-carboxamide,CN1C=CC(=O)C(=C1)C(=O)N,urine,Pubchem,BTO_ID,Aging,MetaboAge


## 3. Check and Fix Duplicate Columns

In [14]:
SOURCE_DFS = [
    ('Metaboage', Metaboage),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[Metaboage] ✓ no duplicates


## 4. Align Schemas and Concatenate

In [15]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 264 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,3845,ChemicalEntity_Tissue,BTO:0000237,ChemicalEntity,<NA>,Tissue,MetaboAge,Pubchem,BTO_ID,4-oxo-1H-quinoline-2-carboxylic acid,cerebrospinal fluid,Aging,Homo sapiens
1,5280450,ChemicalEntity_Tissue,BTO:0001253,ChemicalEntity,<NA>,Tissue,MetaboAge,Pubchem,BTO_ID,"(9Z,12Z)-octadeca-9,12-dienoic acid",skin,Aging,Homo sapiens


## 5. Standardise Fixed-Value Columns

In [16]:
final_df['head_type']  = 'ChemicalEntity'
final_df['tail_type']  = 'Tissue'
final_df['relation']   = 'ChemicalEntity_Tissue'
final_df['tail_id_is'] = 'BTO'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_type:",  set(final_df['head_type']))
print("Unique tail_type:",  set(final_df['tail_type']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'ChemicalEntity_Tissue'}
Unique head_type: {'ChemicalEntity'}
Unique tail_type: {'Tissue'}
Unique kg_source: {'MetaboAge'}


## 6. Deduplicate

In [19]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})

# Derive tail_detail_name from BTO dict
final_df_dedup['tail_detail_name'] = final_df_dedup['tail'].map(BTO_dict)

print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 258 rows


## 7. Add Schema Columns and Enforce Column Order

In [20]:

final_df_dedup = final_df_dedup[REQUIRED_COLS]

print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (258, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,70,ChemicalEntity_Tissue,BTO:0001239,ChemicalEntity,None,Tissue,MetaboAge,Pubchem,BTO,4-methyl-2-oxopentanoic acid,serum,Aging,Homo sapiens
1,127,ChemicalEntity_Tissue,BTO:0000237,ChemicalEntity,None,Tissue,MetaboAge,Pubchem,BTO,2-(4-hydroxyphenyl)acetic acid,cerebrospinal fluid,Aging,Homo sapiens
2,135,ChemicalEntity_Tissue,BTO:0001419,ChemicalEntity,None,Tissue,MetaboAge,Pubchem,BTO,4-hydroxybenzoic acid,urine,Aging,Homo sapiens
3,144,ChemicalEntity_Tissue,BTO:0000237,ChemicalEntity,None,Tissue,MetaboAge,Pubchem,BTO,2-amino-3-(5-hydroxy-1H-indol-3-yl)propanoic acid,cerebrospinal fluid,Aging,Homo sapiens
4,175,ChemicalEntity_Tissue,BTO:0000089,ChemicalEntity,None,Tissue,MetaboAge,Pubchem,BTO,acetate,blood,Aging,Homo sapiens


## 8. QC — NaN Check

In [21]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,258,0,258
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 9. Save Output

In [23]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 258 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/CHEMICALENTITY_TISSUE/ALL_CHEMICALENTITY_TISSUE.csv
